# Giai đoạn 3A: Huấn luyện nền BDD ban ngày

Notebook này chỉ huấn luyện base model trên `data/bdd_day.yaml`.
Phần fine-tune ban đêm đã được tách sang `03b_resume_bdd100k.ipynb` để dễ quản lý và theo dõi từng chặng.

## Mục tiêu
- Sinh run nền ban ngày làm mốc cho stage ban đêm.
- Giữ output và log tách biệt để dễ đối chiếu khi máy bị dừng giữa chừng.
- Dùng warm-start/fine-tune theo chuyển miền day -> night, không trộn hai stage trong cùng một notebook.

## Ràng buộc máy thực tế
- Máy hiện tại: Windows + VSCode, GPU RTX 3050 Laptop 4 GB VRAM.
- Mốc `50 epoch` là có chủ đích, không phải mức hội tụ cuối cùng. Mục đích là giảm rủi ro treo notebook, crash driver, quá nhiệt, dồn standby cache, hoặc mất nguồn khi train liên tục quá lâu.
- Nếu cần train tiếp sau 50 epoch, hãy xem metrics và weight trước rồi mới mở chặng tiếp theo. Không nên để một job rất dài chạy liền trên máy này.

## Điều kiện trước khi chạy
- Chạy `01b_preprocess_bdd100k.ipynb` trước và xác nhận `data/processed/bdd_day` đang tồn tại.
- Không chạy notebook này song song với job train khác.
- Nên đóng bớt ứng dụng nặng GPU/RAM trước khi train.

## Artifact sẽ sinh ra
- `models/runs/bdd_day_base_ep50/`
- `labels.cache` trong cây thư mục label của dataset
- file `*.npy` cạnh image khi dùng `cache='disk'`

Những artifact cache này không phải tài sản lịch sử. Sau khi train/val xong và không cần rerun nhanh nữa, có thể xóa để trả SSD.

## Bộ đếm thời gian
- Notebook có timer tổng để in thời lượng của từng chặng train sau khi `model.train()` kết thúc.

## Chính sách `resume`
- `resume=True` chỉ hợp lý nếu một run đang dở bị ngắt giữa chừng và cần nối tiếp chính run đó từ `last.pt`.
- Notebook này mặc định tạo một run mới, `resume=False`, để giữ logic rõ ràng và tránh nhầm với resume cùng run.


In [ ]:
import sys
import torch
import ultralytics

print(f"Python: {sys.version.split()[0]}")
print(f"Ultralytics: {ultralytics.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA build: {torch.version.cuda}")


In [ ]:
import subprocess
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM khả dụng: {vram_gb:.2f} GB")
    try:
        subprocess.run(["nvidia-smi"], check=False)
    except FileNotFoundError:
        print("nvidia-smi không có sẵn trong PATH")
else:
    print("CUDA không khả dụng; dừng trước khi train.")


## Giải thích setting train
- `batch=8`: mức này đang hợp với VRAM 4 GB hiện tại. Nếu gặp OOM thì hạ xuống `6` hoặc `4` trước, không đổi nhiều tham số cùng lúc.
- `cache='disk'`: ưu tiên giảm đọc ảnh lặp lại. Đổi lại, Ultralytics sẽ sinh `labels.cache` và `*.npy` trên ổ đĩa.
- `workers=4`: giữ mức vừa phải cho Windows laptop. Đẩy quá cao dễ gây áp lực lên I/O và RAM.
- `save_period=5`: máy yếu nên cần checkpoint định kỳ để nếu job chết giữa chừng thì mất ít epoch hơn.
- `patience=15`: tránh kéo train vô ích quá lâu nếu metric đứng lại.

Tên run mặc định của stage này là `bdd_day_base_ep50`. Giữ tên nhất quán để đối chiếu với stage ban đêm.


In [ ]:
from pathlib import Path
import shutil
import time
import torch
from ultralytics import YOLO

BASE_DIR = Path('d:/DAT301m/proposal')
RUNS_DIR = BASE_DIR / 'models' / 'runs'
DAY_YAML = BASE_DIR / 'data' / 'bdd_day.yaml'
DAY_DATA_DIR = BASE_DIR / 'data' / 'processed' / 'bdd_day'

DAY_RUN_NAME = 'bdd_day_base_ep50'
DAY_SOURCE_MODEL = 'yolo11n.pt'

if not torch.cuda.is_available():
    raise RuntimeError('CUDA không khả dụng, dừng trước khi train.')

if not DAY_YAML.exists():
    raise FileNotFoundError(f'Thiếu file YAML cấu hình: {DAY_YAML}')
if not DAY_DATA_DIR.exists():
    raise FileNotFoundError(
        f'Thư mục dữ liệu tiền xử lý bị thiếu: {DAY_DATA_DIR}. Chạy lại 01b_preprocess_bdd100k.ipynb trước.'
    )

RUNS_DIR.mkdir(parents=True, exist_ok=True)

start_time = time.time()
start_label = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(start_time))
print(f'Bắt đầu train lúc: {start_label}')

model = YOLO(DAY_SOURCE_MODEL)
try:
    results = model.train(
        data=str(DAY_YAML),
        epochs=50,
        patience=15,
        batch=8,
        cache='disk',
        imgsz=640,
        device=0,
        workers=4,
        optimizer='auto',
        mosaic=0.5,
        close_mosaic=10,
        save_period=5,
        seed=0,
        project=str(RUNS_DIR),
        name=DAY_RUN_NAME,
        exist_ok=False,
        resume=False,
    )
finally:
    end_time = time.time()
    elapsed_seconds = end_time - start_time
    end_label = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(end_time))
    print(f'Kết thúc lúc: {end_label}')
    print(f'Thời gian chạy: {elapsed_seconds / 60:.1f} phút ({elapsed_seconds / 3600:.2f} giờ)')

run_dir = Path(model.trainer.save_dir)
best_pt = run_dir / 'weights' / 'best.pt'
tagged_pt = run_dir / 'weights' / f'{run_dir.name}.pt'

if best_pt.exists():
    shutil.copy2(best_pt, tagged_pt)
    print(f'Đã sao chép trọng số tốt nhất tới {tagged_pt}')
else:
    print(f'Không tìm thấy trọng số tốt nhất tại {best_pt}')

print('\n=== TÓM TẮT ===')
print(f'Thư mục run: {run_dir}')
print(f'Trọng số tốt nhất đã gắn nhãn: {tagged_pt}')
print('Nếu máy bị ngắt giữa chừng trước mốc 50 epoch, chỉ khi đó mới cần resume cùng run từ `last.pt`.')
